In [1]:
using Pkg                # Load Julia's built-in package manager module (Pkg)

Pkg.activate("..")       # Activate the Project.toml located one directory above
                         # (Allows notebooks/ to use the repository root environment)

using Revise             # Development tool that automatically reloads changes
                         # made to files in src/ without restarting Julia

using Horizons3DViewer2  # Load the custom package Horizons3DViewer2
                         # (This loads src/Horizons3DViewer2.jl and its modules)

using EphemerisSources   # Tools for accessing planetary ephemeris data
using AstroTime, Dates   # Time systems and date handling utilities
using GLMakie            # High-performance interactive visualization backend
using ProgressMeter      # Progress bar utilities for long computations
using LinearAlgebra      # Standard linear algebra operations (norm, dot, etc.)

  Activating project at `~/work/Horizons3DViewer2`


https://juliaastro.org/EphemerisSources.jl/docs/stable/examples/telescope-study/
https://juliaastro.org/AstroTime/stable/tutorial/#Tutorial
https://ssd.jpl.nasa.gov/horizons/app.html#/

In [2]:
######################################
# Specify the data to be retrieved
######################################

frame = "J2000"   # Input parameter for the NASA Horizons API
wrt   = "sun"     # Input parameter for the NASA Horizons API
units = "KM-S"    # Input parameter for the NASA Horizons API

####################
#spacecraft = "Orion EM-1"
#start = TDBEpoch(2022, 11, 16, 9, 4, 0.0)
#stop  = TDBEpoch(2022, 12, 11, 17, 21, 0.0)
#step  = "20 min"
#step_size_sec = 20*60

####################
spacecraft = "BepiColombo"                   # Input parameter for the NASA Horizons API
start = TDBEpoch(2018, 10, 20, 2, 13, 29.0)  # Input parameter for the NASA Horizons API
stop  = TDBEpoch(2026, 11, 20, 0, 0, 0.0)    # Input parameter for the NASA Horizons API
step  = "12 hour"                            # Input parameter for the NASA Horizons API
step_size_sec = 12*60*60                     # Used in the function `propagate_future_trajectories_at_each_epoch_for_animation`

####################
#spacecraft = "BepiColombo"
#start = TDBEpoch(2026, 11, 20, 0, 0, 0.0)
#stop  = TDBEpoch(2027, 4, 7, 0, 1, 9.0)
#step  = "5 min"
#step_size_sec = 5*60

####################
#spacecraft = "Orion EM-1"
#start = TDBEpoch(2022, 11, 16, 9, 3, 0.0)
#stop  = TDBEpoch(2022, 11, 20, 0, 0, 0.0)
#step  = "10 min"
#step_size_sec = 10*60

####################
#spacecraft = "SLIM"
#start = TDBEpoch(2023, 9, 7, 0, 31, 51.0)
#stop  = TDBEpoch(2023, 12, 26, 0, 31, 50.0)
#step  = "15 min"
#step_size_sec = 15*60

####################
#spacecraft = "JUICE"
#start = TDBEpoch(2023, 4, 14, 12, 43, 27.0)
#stop  = TDBEpoch(2031, 7, 21, 6, 31, 30.0)
#step  = "1 day"
#step_size_sec = 24*60*60

####################
#spacecraft = "CAPSTONE"
#start = TDBEpoch(2026, 1, 25, 3, 54, 1.0)
#stop  = TDBEpoch(2026, 3,  3, 11, 36, 27.0)
#stop  = start + 150days
#step  = "10 min"
#step_size_sec = 10*60

####################
#spacecraft = "Hayabusa 2"
#start = TDBEpoch(2014, 12, 3, 6, 14, 54.0)
#stop  = TDBEpoch(2026, 4, 10, 0, 0, 0.0)
#step  = "24 hour"
#step_size_sec = 24*60*60
#ryugu_horizons = ephemeris("162173", start, stop, step ; wrt = wrt, units = units, frame = frame)
#ryugu_states = horizons_to_matrix(ryugu_horizons)

####################
#spacecraft = "Europa Clipper" #全体
#start = TDBEpoch(2024, 10, 14, 16, 15, 4.0)
#stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
#step  = "6 hour"
#step_size_sec = 6*60*60
#europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
#europa_states = horizons_to_matrix(europa_horizons)

####################
#spacecraft = "Europa Clipper" # 木星に入ってから
#start = TDBEpoch(2030, 4, 1, 0, 0, 0.0)
#start = TDBEpoch(2031, 1, 1, 0, 0, 0.0)
#stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
#stop  = TDBEpoch(2032, 1, 1, 0, 0, 0.0)
#step  = "1 hour"
#step_size_sec = 1*60*60
#europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
#europa_states = horizons_to_matrix(europa_horizons)


43200

In [3]:
############################################################################
# Retrieve ephemeris data and convert it into a convenient internal format
############################################################################

# Retrieve ephemeris data from JPL Horizons
sun_horizons     = ephemeris("sun", start, stop, step; wrt = wrt, units = units, frame = frame)
earth_horizons   = ephemeris("earth", start, stop, step; wrt = wrt, units = units, frame = frame)
mars_horizons    = ephemeris("mars", start, stop, step; wrt = wrt, units = units, frame = frame)
jupiter_horizons = ephemeris("jupiter", start, stop, step; wrt = wrt, units = units, frame = frame)
moon_horizons    = ephemeris("moon", start, stop, step; wrt = wrt, units = units, frame = frame)
venus_horizons   = ephemeris("venus", start, stop, step; wrt = wrt, units = units, frame = frame)
mercury_horizons = ephemeris("mercury", start, stop, step; wrt = wrt, units = units, frame = frame)
sc1_horizons     = ephemeris(spacecraft, start, stop, step; wrt = wrt, units = units, frame = frame)

# Convert the Horizons outputs into 6×N state matrices
sun_states       = horizons_to_matrix(sun_horizons)
earth_states     = horizons_to_matrix(earth_horizons)
mars_states      = horizons_to_matrix(mars_horizons)
jupiter_states   = horizons_to_matrix(jupiter_horizons)
moon_states      = horizons_to_matrix(moon_horizons)
venus_states     = horizons_to_matrix(venus_horizons)
mercury_states   = horizons_to_matrix(mercury_horizons)
sc1_states       = horizons_to_matrix(sc1_horizons)

# Convert Horizons time stamps (Julian Date) to TDBEpoch values for animation labels
t_tdb = TDBEpoch.(sc1_horizons.t .* days; origin = :julian)

# Diagnostic output (for sanity checks)
println(size(sc1_states))
println(sc1_states[1:6, 1])
println(t_tdb[1])

(6, 5906)
[1.33455923459285e8, 6.621153691250275e7, -6703.55413742736, -24.09834551510919, 28.19970439297871, -1.843049930526249]
2018-10-20T02:13:28.999 TDB


In [4]:
######################################
# Heliocentric specific energy
######################################

sc1_heliocentric_energy     = calc_heliocentric_specific_energy(sc1_states)
mercury_heliocentric_energy = calc_heliocentric_specific_energy(mercury_states)
earth_heliocentric_energy   = calc_heliocentric_specific_energy(earth_states)

######################################
# Coordinate transformations
######################################

# Transform states to the EME2000 frame
# (Note: In practice, EME2000 is equivalent to the J2000 mean equator and equinox frame.
#  In SPICE and most astrodynamics applications, the J2000 frame is treated as identical
#  to EME2000, and any differences are negligible for typical spacecraft trajectory analyses.)
moon_states_eme = eclJ2000_to_EME2000(moon_states - earth_states)
sc1_states_eme  = eclJ2000_to_EME2000(sc1_states  - earth_states)

# Transform to the Earth–Moon line-fixed rotating frame (Earth-centered)
moon_em_fixed_e = make_line_fixed_frame(earth_states, moon_states, moon_states)
sc1_em_fixed_e  = make_line_fixed_frame(earth_states, moon_states, sc1_states)
sun_em_fixed_e  = make_line_fixed_frame(earth_states, moon_states, sun_states)

# Transform to the Earth–Moon line-fixed rotating frame (Moon-centered)
sc1_em_fixed_m  = sc1_em_fixed_e - moon_em_fixed_e
sun_em_fixed_m  = sun_em_fixed_e - moon_em_fixed_e

# Transform to the Sun–Earth line-fixed rotating frame
earth_se_fixed_s = make_line_fixed_frame(sun_states, earth_states, earth_states)
moon_se_fixed_e  = make_line_fixed_frame(sun_states, earth_states, moon_states) - earth_se_fixed_s
sc1_se_fixed_e   = make_line_fixed_frame(sun_states, earth_states, sc1_states)  - earth_se_fixed_s

# Transform to the lunar orbital plane–fixed frame (LOP: Lunar Orbital Plane)
# Here, the LOP is defined from the Moon’s orbital plane at the initial epoch and is kept fixed for all subsequent epochs.
moon_lop_e = rotate_to_plane_fixed_t0(earth_states, moon_states, moon_states)
sc1_lop_e  = rotate_to_plane_fixed_t0(earth_states, moon_states, sc1_states)
sun_lop_e  = rotate_to_plane_fixed_t0(earth_states, moon_states, sun_states)

3×5906 Matrix{Float64}:
 -1.32971e8  -1.32372e8  -1.31764e8  …  -8.25103e7  -8.14226e7  -8.03287e7
 -6.59019e7  -6.70467e7  -6.81865e7     -1.22146e8  -1.22859e8  -1.23562e8
 -1.30257e7  -1.30342e7  -1.30418e7     -1.18626e7  -1.18143e7  -1.17651e7

In [ ]:
######################################
# Animation settings
######################################

# Whether to render the animation
animation_flag = 1

# Length of the trajectory trail
trail_len_long  = 10000
trail_len_short = 500    # Mio in Sun-centered coordinates
#trail_len_short = 5000  # Mio in Mercury-centered coordinates

# Whether to propagate future trajectories for animation
show_propagated_flag = 0

# Animation frame selection
sampling_interval = 5    # Mio in Sun-centered coordinates
#sampling_interval = 50  # Mio in Mercury-centered coordinates
N      = size(sc1_states, 2)
frames = 1:sampling_interval:(N-1)

# Frame switching (use finer resolution after this point)
#switch_frame = Int(floor(0.8 * N))   # Example: use finer sampling for the last 20%
#frames1 = 1:sampling_interval1:(switch_frame-1)
#frames2 = switch_frame:sampling_interval2:(N-1)
#frames  = vcat(frames1, frames2)

######################################
# Plotting area settings
######################################

set_theme!(theme_black())
update_theme!(fontsize= 18)

fig1 = Figure(size = (900, 600))
fig2 = Figure(size = (900, 600))
fig3 = Figure(size = (900, 600))
fig4 = Figure(size = (900, 600))
fig5 = Figure(size = (900, 600))
fig6 = Figure(size = (900, 600))
fig7 = Figure(size = (900, 600))
fig8 = Figure(size = (900, 600))
fig9 = Figure(size = (900, 600))

ax_ecliptic_s      = default_axis3(fig1[1,1:3]; scale=10e7,  title="")
ax_ecliptic_j      = default_axis3(fig2[1, 1]; scale=3e6,  title="Ecliptic of J2000, Jupiter centered")
ax_ecliptic_e      = default_axis3(fig3[1, 1]; scale=3e4,  title="Ecliptic of J2000, Earth centered")
ax_ecliptic_mer    = default_axis3(fig9[1, 1]; scale=2e4,  azimuth = -1.1pi, elevation = 0.05pi, title="Mercury")

ax_lop_e           = default_axis3(fig4[1, 1]; scale=3e5,  title="Inertial frame")
ax_eme_e           = default_axis3(fig5[1, 1]; scale=2e5,  title="J2000")

ax_em_fixed_m      = default_axis3(fig6[1, 1]; scale=30e3, azimuth = -1.0pi, elevation = -0pi, title="Earth-Moon line fixed rotational frame, Moon centered")
ax_em_fixed_e      = default_axis3(fig7[1, 1]; scale=2e5,  offset=(2e5, 0.0, 0.0), title="Earth-Moon line fixed rotational frame, Earth centered")
ax_se_fixed_e      = default_axis3(fig8[1, 1];             title="Sun-Earth line fixed rotational frame, Earth centered")

ax_energy          = Axis(fig1[1, 4], xticks = (1:1, ["Heliocentric energy"]),title = "Heliocentric energy")

hidedecorations!(ax_energy)
limits!(ax_energy, (0,2),(0,earth_heliocentric_energy[1]+1250+100))

######################################
# Common plotting routines for both static images and animation
######################################

sun_sphere     = Sphere(Point3f(0.0, 0.0, 0.0), 696340)
earth_sphere   = Sphere(Point3f(0.0, 0.0, 0.0), 6378)
moon_sphere    = Sphere(Point3f(0.0, 0.0, 0.0), 1737)
jupiter_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 69911)
mercury_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 2440)
europa_sphere  = Sphere(Point3f(0.0, 0.0, 0.0), 1600)

# Sun
scatter!(ax_ecliptic_s, 0.0, 0.0, 0.0, markersize=15, color=:white)
text!(ax_ecliptic_s, 0.0, 0.0, 0.0,text = "Sun",color = :white,fontsize = 18)

# Earth
mesh!(ax_ecliptic_e, earth_sphere, color=:cyan, shading=true)
mesh!(ax_eme_e, earth_sphere, color=:cyan)
mesh!(ax_lop_e, earth_sphere, color=:cyan, shading=true)
mesh!(ax_se_fixed_e, earth_sphere, color=:cyan)
mesh!(ax_em_fixed_e, earth_sphere, color=:cyan, shading=true)
# Moon
mesh!(ax_em_fixed_m, moon_sphere, color=:yellow, shading=true)
# Jupiter
mesh!(ax_ecliptic_j, jupiter_sphere, color=:orange)
# Mercury
mesh!(ax_ecliptic_mer, mercury_sphere, color=:white, shading=true)


######################################
# Render static image or animation
######################################

if animation_flag == 0

    lines!(ax_ecliptic_s, earth_states[1:3, :], linewidth=1, color=:cyan)
    #lines!(ax_ecliptic_s, mars_states[1:3, :], linewidth=1, color=:red)
    #lines!(ax_ecliptic_s, jupiter_states[1:3, :], linewidth=1, color=:orange)
    lines!(ax_ecliptic_s, venus_states[1:3, :], linewidth=1, color=:orange)
    lines!(ax_ecliptic_s, mercury_states[1:3, :], linewidth=1, color=:magenta)

    lines!(ax_ecliptic_s, sc1_states[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_ecliptic_mer, sc1_states[1:3, :]-mercury_states[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_ecliptic_e, moon_states[1:3, :]- earth_states[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :],  linewidth=1, color=:lime)
    
    lines!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:lime)
    #lines!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:white)

    lines!(ax_eme_e, moon_states_eme[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_eme_e, sc1_states_eme[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_lop_e, moon_lop_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_lop_e, sc1_lop_e[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_se_fixed_e, moon_se_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_se_fixed_e, sc1_se_fixed_e[1:3, :],  linewidth=1, color=:lime)
    
    lines!(ax_em_fixed_e, moon_em_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_em_fixed_e, sc1_em_fixed_e[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_em_fixed_m, sc1_em_fixed_m[1:3, :],  linewidth=1, color=:lime)

else
    frame_0   = Observable(0)
    frame_0[] = 1

    if show_propagated_flag == 1
        
        results_dict = propagate_future_trajectories_at_each_epoch_for_animation(
            sc1_states, earth_states, moon_states, mars_states, jupiter_states;
            sampling_interval = sampling_interval,
            step_size_sec     = step_size_sec,
            reltol            = 1e-10,
            abstol            = 1e-10,
            show_progress     = true,
        )
        
        lines!(ax_ecliptic_s, @lift(results_dict[$frame_0].sol[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_lop_e, @lift(results_dict[$frame_0].sol_lop_e[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_em_fixed_e, @lift(results_dict[$frame_0].sol_em_fixed_e[1:3, :]),  linewidth=1, color=:red)
        #lines!(ax_em_fixed_m, @lift(results_dict[$frame_0].sol_em_fixed_m[1:3, :]),  linewidth=1, color=:red)
    end

    text!(ax_ecliptic_s, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative)
    text!(ax_ecliptic_mer, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative)
    text!(ax_em_fixed_m, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")* " TDB"), offset=(0, 0), align=(:right, :top), space=:relative)

    text!(ax_ecliptic_s,@lift(Point3f(earth_states[1, $frame_0],earth_states[2, $frame_0],earth_states[3, $frame_0])),text = "Earth",color = :cyan,fontsize = 18)
    #text!(ax_ecliptic_s,@lift(Point3f(venus_states[1, $frame_0],venus_states[2, $frame_0],venus_states[3, $frame_0])),text = "Venus",color = :orange,fontsize = 18)
    #text!(ax_ecliptic_s,@lift(Point3f(mercury_states[1, $frame_0],mercury_states[2, $frame_0],mercury_states[3, $frame_0])),text = "Mercury",color = :white,fontsize = 18)

    #energy
    barplot!(ax_energy,1,@lift(sc1_heliocentric_energy[$frame_0] + 1250),color = :lime )
    lines!(ax_energy,[0,2],[mercury_heliocentric_energy[1]+1250,mercury_heliocentric_energy[1]+1250],color = :white)
    lines!(ax_energy,[0,2],[earth_heliocentric_energy[1]+1250,earth_heliocentric_energy[1]+1250],color = :cyan)
    text!(ax_energy,0.03, mercury_heliocentric_energy[1] + 1250,text = "Mercury",color = :white,fontsize = 15,align = (:left, :bottom))
    text!(ax_energy,0.03, earth_heliocentric_energy[1] + 1250,  text = "Earth",  color = :cyan, fontsize = 15,align = (:left, :bottom))
    
    traj_fade!(ax_ecliptic_s, earth_states[1:3, :], frame_0; trail_len=trail_len_short, color=:cyan, markersize = 15)
    traj_fade!(ax_ecliptic_s, venus_states[1:3, :], frame_0; trail_len=trail_len_short, color=:orange, markersize = 15)
    traj_fade!(ax_ecliptic_s, mercury_states[1:3, :], frame_0; trail_len=trail_len_short, color=:white, markersize = 15)
    #traj_fade!(ax_ecliptic_s, mars_states[1:3, :], frame_0; trail_len=trail_len_long, color=:pink)
    #traj_fade!(ax_ecliptic_s, jupiter_states[1:3, :], frame_0; trail_len=trail_len_long, color=:orange)
    traj_fade!(ax_ecliptic_s, sc1_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)

    traj_fade!(ax_ecliptic_mer, sc1_states[1:3, :] - mercury_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime, markersize = 0)

    traj_fade!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    traj_fade!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    #traj_fade!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:white)

    traj_fade!(ax_lop_e, moon_lop_e, frame_0; trail_len=trail_len_short, color=:yellow)    
    traj_fade!(ax_lop_e, sc1_lop_e, frame_0; trail_len=trail_len_short, color=:lime)

    traj_fade!(ax_em_fixed_e, sc1_em_fixed_e[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    traj_fade!(ax_em_fixed_e, moon_em_fixed_e[1:3, :], frame_0; trail_len=trail_len_short, color=:yellow)

    traj_fade!(ax_em_fixed_m, sc1_em_fixed_m[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)

    p = Progress(length(frames); desc = "Rendering movie")

    # Export animation
    record(fig1, "../output/$(spacecraft)_test.mp4", frames; framerate = 30) do frame
        # framerate = number of frames per second (60 fps is typically the practical upper limit)
        # Update planetary frame index for drawing Earth/Mars trajectories

        frame_0[] = frame
        
        # Update solar illumination for each sphere
        set_sun_light!(ax_em_fixed_m, sun_em_fixed_m[1:3, frame])
        set_sun_light!(ax_em_fixed_e, sun_em_fixed_e[1:3, frame])
        set_sun_light!(ax_lop_e, sun_lop_e[1:3, frame])
        set_sun_light!(ax_ecliptic_mer, sun_states[1:3, frame] - mercury_states[1:3, frame])
        
        next!(p)  # Advance progress bar
    end

end

fig1


Propagating trajectories 100%|███████████████████████████| Time: 0:00:20
Rendering movie 100%|████████████████████████████████████| Time: 0:00:28
